# REST API: Alpha Vantage Tesla News

In this notebook, Tesla-related financial news are collected from the Alpha Vantage News & Sentiment REST API.  
The API data is cleaned and saved as structured CSV files for later merging with Tesla stock data and scraped news data.

# 2. Imports

In [1]:
import pandas as pd
import requests
import os

# 3. API Request

In [2]:
api_key = "3PHM0X2BD5ONBOBW"

url = (
    "https://www.alphavantage.co/query"
    "?function=NEWS_SENTIMENT"
    "&tickers=TSLA"
    "&sort=LATEST"
    "&limit=100"
    f"&apikey={api_key}"
)

response = requests.get(url, timeout=30)

print("Status code:", response.status_code)

data = response.json()

print("Response keys:", data.keys())

Status code: 200
Response keys: dict_keys(['items', 'sentiment_score_definition', 'relevance_score_definition', 'feed'])


# 4. Check API Response

In [3]:
if "feed" in data:
    articles = data["feed"]
    print("Number of articles:", len(articles))
else:
    print("No feed found.")
    print(data)

Number of articles: 50


# 5. Convert to DataFrame

In [4]:
api_raw_df = pd.DataFrame(articles)

print("Shape:", api_raw_df.shape)
print("Columns:")
print(api_raw_df.columns)

display(api_raw_df.head())

Shape: (50, 13)
Columns:
Index(['title', 'url', 'time_published', 'authors', 'summary', 'banner_image',
       'source', 'category_within_source', 'source_domain', 'topics',
       'overall_sentiment_score', 'overall_sentiment_label',
       'ticker_sentiment'],
      dtype='object')


,title,url,time_published,authors,summary,banner_image,source,category_within_source,source_domain,topics,overall_sentiment_score,overall_sentiment_label,ticker_sentiment
0,UBS reiterates Packaging Corp. of America stoc...,https://ca.investing.com/news/stock-market-new...,20260611T143829,[Investing.com],UBS has reiterated a Buy rating and a $248.00 ...,https://i-invdn-com.investing.com/news/LYNXMPE...,Investing.com Canada,General,Investing.com Canada,"[{'topic': 'earnings', 'relevance_score': '1.0...",0.049732,Neutral,"[{'ticker': 'PKG', 'relevance_score': '1.00000..."
1,Oppenheimer raises Tesla stock price target on...,https://www.investing.com/news/analyst-ratings...,20260611T132540,[Investing.com],Oppenheimer has increased its price target for...,https://i-invdn-com.investing.com/news/moved_L...,Investing.com,General,Investing.com,"[{'topic': 'financial_markets', 'relevance_sco...",0.203912,Somewhat-Bullish,"[{'ticker': 'TSLA', 'relevance_score': '1.0000..."
2,Is the 2026 Toyota bZ Woodland a Good Crossove...,https://www.cars.com/articles/is-the-2026-toyo...,20260611T113909,[Jim Travers],The 2026 Toyota bZ Woodland is an electric cro...,https://images.cars.com/cldstatic/wp-content/u...,Cars.com,General,Cars.com,"[{'topic': 'energy_transportation', 'relevance...",0.037104,Neutral,"[{'ticker': 'CARS', 'relevance_score': '0.3334..."
3,"General Motors pivots battery strategy, puts L...",https://seekingalpha.com/news/4602466-general-...,20260611T092702,"[Manshi Mamtora, CFA]",General Motors is reportedly shifting its elec...,https://static.seekingalpha.com/cdn/s3/uploads...,Seeking Alpha,General,Seeking Alpha,"[{'topic': 'energy_transportation', 'relevance...",0.138295,Neutral,"[{'ticker': 'GM', 'relevance_score': '1.000000..."
4,Musk to discuss Terafab chip plant plans at AS...,https://seekingalpha.com/news/4602459-musk-to-...,20260611T081040,[Preeti Singh],Elon Musk is scheduled to virtually attend ASM...,https://static.seekingalpha.com/cdn/s3/uploads...,Seeking Alpha,General,Seeking Alpha,"[{'topic': 'technology', 'relevance_score': '0...",0.172115,Somewhat-Bullish,"[{'ticker': 'ASML', 'relevance_score': '1.0000..."


# 6. Select Relevant Columns

In [5]:
selected_columns = [
    "title",
    "url",
    "time_published",
    "summary",
    "source",
    "overall_sentiment_score",
    "overall_sentiment_label"
]

api_cleaned_df = api_raw_df[selected_columns].copy()

display(api_cleaned_df.head())

,title,url,time_published,summary,source,overall_sentiment_score,overall_sentiment_label
0,UBS reiterates Packaging Corp. of America stoc...,https://ca.investing.com/news/stock-market-new...,20260611T143829,UBS has reiterated a Buy rating and a $248.00 ...,Investing.com Canada,0.049732,Neutral
1,Oppenheimer raises Tesla stock price target on...,https://www.investing.com/news/analyst-ratings...,20260611T132540,Oppenheimer has increased its price target for...,Investing.com,0.203912,Somewhat-Bullish
2,Is the 2026 Toyota bZ Woodland a Good Crossove...,https://www.cars.com/articles/is-the-2026-toyo...,20260611T113909,The 2026 Toyota bZ Woodland is an electric cro...,Cars.com,0.037104,Neutral
3,"General Motors pivots battery strategy, puts L...",https://seekingalpha.com/news/4602466-general-...,20260611T092702,General Motors is reportedly shifting its elec...,Seeking Alpha,0.138295,Neutral
4,Musk to discuss Terafab chip plant plans at AS...,https://seekingalpha.com/news/4602459-musk-to-...,20260611T081040,Elon Musk is scheduled to virtually attend ASM...,Seeking Alpha,0.172115,Somewhat-Bullish


# 7. Clean Data

In [6]:
api_cleaned_df = api_cleaned_df.dropna(subset=["title", "url", "time_published"])

api_cleaned_df["title"] = api_cleaned_df["title"].astype(str).str.strip()
api_cleaned_df["url"] = api_cleaned_df["url"].astype(str).str.strip()
api_cleaned_df["summary"] = api_cleaned_df["summary"].fillna("").astype(str).str.strip()
api_cleaned_df["source"] = api_cleaned_df["source"].fillna("Unknown").astype(str).str.strip()

api_cleaned_df["time_published"] = pd.to_datetime(
    api_cleaned_df["time_published"],
    format="%Y%m%dT%H%M%S",
    errors="coerce"
)

api_cleaned_df = api_cleaned_df.dropna(subset=["time_published"])

api_cleaned_df["date"] = api_cleaned_df["time_published"].dt.date

api_cleaned_df["overall_sentiment_score"] = pd.to_numeric(
    api_cleaned_df["overall_sentiment_score"],
    errors="coerce"
)

api_cleaned_df = api_cleaned_df.drop_duplicates(subset=["title", "url", "time_published"])

print("Cleaned shape:", api_cleaned_df.shape)
display(api_cleaned_df.head())

Cleaned shape: (50, 8)


,title,url,time_published,summary,source,overall_sentiment_score,overall_sentiment_label,date
0,UBS reiterates Packaging Corp. of America stoc...,https://ca.investing.com/news/stock-market-new...,2026-06-11 14:38:29,UBS has reiterated a Buy rating and a $248.00 ...,Investing.com Canada,0.049732,Neutral,2026-06-11
1,Oppenheimer raises Tesla stock price target on...,https://www.investing.com/news/analyst-ratings...,2026-06-11 13:25:40,Oppenheimer has increased its price target for...,Investing.com,0.203912,Somewhat-Bullish,2026-06-11
2,Is the 2026 Toyota bZ Woodland a Good Crossove...,https://www.cars.com/articles/is-the-2026-toyo...,2026-06-11 11:39:09,The 2026 Toyota bZ Woodland is an electric cro...,Cars.com,0.037104,Neutral,2026-06-11
3,"General Motors pivots battery strategy, puts L...",https://seekingalpha.com/news/4602466-general-...,2026-06-11 09:27:02,General Motors is reportedly shifting its elec...,Seeking Alpha,0.138295,Neutral,2026-06-11
4,Musk to discuss Terafab chip plant plans at AS...,https://seekingalpha.com/news/4602459-musk-to-...,2026-06-11 08:10:40,Elon Musk is scheduled to virtually attend ASM...,Seeking Alpha,0.172115,Somewhat-Bullish,2026-06-11


# 8. Check Missing Values

In [7]:
api_cleaned_df.isnull().sum()

title                      0
url                        0
time_published             0
summary                    0
source                     0
overall_sentiment_score    0
overall_sentiment_label    0
date                       0
dtype: int64

# 9. Save Data

In [8]:
os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

api_raw_df.to_csv("../data/raw/alpha_vantage_tesla_news_raw.csv", index=False)
api_cleaned_df.to_csv("../data/processed/alpha_vantage_tesla_news_cleaned.csv", index=False)

print("Saved raw data to ../data/raw/alpha_vantage_tesla_news_raw.csv")
print("Saved cleaned data to ../data/processed/alpha_vantage_tesla_news_cleaned.csv")
print("Final shape:", api_cleaned_df.shape)

Saved raw data to ../data/raw/alpha_vantage_tesla_news_raw.csv
Saved cleaned data to ../data/processed/alpha_vantage_tesla_news_cleaned.csv
Final shape: (50, 8)


# 10. Quick Analysis

In [9]:
print("Date range:")
print(api_cleaned_df["date"].min(), "to", api_cleaned_df["date"].max())

print("\nSentiment label counts:")
print(api_cleaned_df["overall_sentiment_label"].value_counts())

print("\nAverage sentiment score:")
print(api_cleaned_df["overall_sentiment_score"].mean())

Date range:
2026-06-09 to 2026-06-11

Sentiment label counts:
overall_sentiment_label
Neutral             25
Somewhat-Bullish    15
Somewhat-Bearish     4
Bearish              3
Bullish              3
Name: count, dtype: int64

Average sentiment score:
0.081795
